In [1]:
from pathlib import Path
import datetime as dt

import numpy as np
import pandas as pd 

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.transforms import v2
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader, Subset


from custom_tools.tools import frame_video_split
from custom_tools.custom_logger import create_logger
from custom_tools.datasets import CADICADataset
from custom_tools.train_tools import befor_train_init, train_loop, compute_metrics
from custom_tools.architecture.ResNet_18_backbone import ResNet_18_Backbone

**LCA**

Сплит на **уровне видео** (одно видео пациента идет или в трейн или в тест).

Кросс-валидация не выполнялась

Две стадии обучения:
1) Заменяем последний слой - обучаем с большим LR (1e-3)
2) Берем лучший checkpoint, размораживаем всю сеть и обучаем с маленьким LR (1e-5)

В принципе, после первого этапа обучения, почти получили метрики из [статьи](https://arxiv.org/pdf/2402.00570) для данного типа артерии (LCA), для данного размера пакета и данной архитектуры сети (ResNet-18)

---
Увеличить количество эпох на первом этапе обучения (до 20)

**Увеличение количества эпох не принесло результата**

Второй этап не производился

---

Из статьи:
* Accuracy: **0.712**
* Balanced Accuracy: **0.656**
* F1: **0.794**

Метрики после первого этапа:
* Accuracy: 
* Balanced Accuracy: 
* F1: 

In [2]:
# ===================== PARAMETERS =====================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# set seed
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

PATH_SELECTED_VIDEO = Path("../../datasets/CADICA/selectedVideos")
frame_cadica = pd.read_csv("../data/cadica_dataset.csv")

TRAIN_SIZE_DATA = 0.8
BATCH_SIZE = 64
input_img_shape = (3,224,224)

class_weights = [2.2, 1]
weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

LR = 1e-3
MOMENTUM = 0.9
LAMBDA_LR = 1e-4

base_dir_for_model_save = "../models_checkpoints" # dir for models save

model_name = "resnet_18_backbone" # model train name
prefix_train = "classifier_layer_train_v2" # prefix train



# ============ comment for log ============
comment = f""" 
===== general parameters =====
SEED: {SEED}
DEVICE: {DEVICE}

PATH_SELECTED_VIDEO: {PATH_SELECTED_VIDEO}

TRAIN_SIZE_DATA: {TRAIN_SIZE_DATA}
BATCH_SIZE: {BATCH_SIZE}
input_img_shape: {input_img_shape}

===== train parameters =====
class_weights: {class_weights}
weights: {weights}

LR: {LR}
MOMENTUM: {MOMENTUM}
LAMBDA_LR: {LAMBDA_LR}

===== about model and save =====
base_dir_for_model_save: {base_dir_for_model_save}

model_name: {model_name}
prefix_train: {prefix_train}

===== comments =====
LCA
split video level
SGDM optimizer
use augmentation
use ImageNet stats for (mean, std) normalization 
"""

## Load and split frame

In [3]:
# === разбиение на LCA и RCA === 
lca_cadica_frame = frame_cadica[frame_cadica['artery'] == 'LCA']
# rca_cadica_frame = frame_cadica[frame_cadica['artery'] == 'RCA']

# === split LCA and RCA frames for train/test ===
frame_lca_train, frame_lca_test = frame_video_split(lca_cadica_frame, "video_id", TRAIN_SIZE_DATA)
frame_lca_train, frame_lca_valid = frame_video_split(frame_lca_train, "video_id", TRAIN_SIZE_DATA)

# === для преобразований ===
train_transform = v2.Compose([
    v2.RandomAffine(
        degrees=25,                          
        translate=(25/512, 25/512),          
        scale=(0.8, 1.7),                    
        interpolation=InterpolationMode.BILINEAR,
        fill=0                               
    ),
    v2.Resize((224, 224)),               
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True), # Преобразование типа, scale=True (масштабирование значений от 0 до 1)
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Нормализация от ImageNet
])

test_transform = v2.Compose([
    v2.Resize((224, 224)),               
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
# ==== weights calculate ====
# class_0 = frame_lca_train[frame_lca_train['label'] == 0].shape[0]
# class_1 = frame_lca_train[frame_lca_train['label'] == 1].shape[0]
# total = class_0 + class_1
# weight_0 = class_1 / class_0
# weight_1 = 1
# print(class_0, class_1, total)
# print(weight_0, weight_1)

In [5]:
# === datasets create ===
# LCA
train_lca_dataset = CADICADataset(frame_lca_train, "image_path", "label", transform=train_transform)
valid_lca_dataset = CADICADataset(frame_lca_valid, "image_path", "label", transform=test_transform)
test_lca_dataset = CADICADataset(frame_lca_test, "image_path", "label", transform=test_transform)

In [6]:
# === dev set create ===
indices = np.random.choice(len(train_lca_dataset), size=64)
dev_dataset = Subset(train_lca_dataset, indices)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=1
)

In [7]:
# === create loaders ===
train_loader = DataLoader(
    train_lca_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=1
)
valid_loader = DataLoader(
    valid_lca_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=1
)
test_loader = DataLoader(
    test_lca_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=1
)

In [8]:
# model init
model_resnet = ResNet_18_Backbone(input_img_shape, out_classes=2, freeze_backbone=True).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.SGD(model_resnet.parameters(), lr=LR, weight_decay=LAMBDA_LR, momentum=MOMENTUM)
# optimizer = optim.RMSprop(model_resnet.parameters(), lr=LR, weight_decay=LAMBDA_LR)

In [9]:
total_params = sum(p.numel() for p in model_resnet.parameters())
trainable_params = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)

print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых:       {trainable_params:,}")

Всего параметров: 11,177,538
Обучаемых:       1,026


In [10]:
time_now = dt.datetime.now().strftime("%Y-%m-%d_%H_%M_%S")

path2save = befor_train_init(
    base_dir_for_model_save,
    model_name,
    time_now,
    prefix_train
)

# === logger create ===
train_logger = create_logger(path2save, "train_log.log")

# === comment write log ===
train_logger.info(comment)

2026-04-09 01:07:22,697 - INFO -  
===== general parameters =====
SEED: 42
DEVICE: cuda

PATH_SELECTED_VIDEO: ..\..\datasets\CADICA\selectedVideos

TRAIN_SIZE_DATA: 0.8
BATCH_SIZE: 64
input_img_shape: (3, 224, 224)

===== train parameters =====
class_weights: [2.2, 1]
weights: tensor([2.2000, 1.0000], device='cuda:0')

LR: 0.001
MOMENTUM: 0.9
LAMBDA_LR: 0.0001

===== about model and save =====
base_dir_for_model_save: ../models_checkpoints

model_name: resnet_18_backbone
prefix_train: classifier_layer_train_v2

===== comments =====
LCA
split video level
SGDM optimizer
use augmentation
use ImageNet stats for (mean, std) normalization 



In [11]:
train_history = train_loop(
    model_resnet, 
    loss_fn, 
    optimizer, 
    train_loader, 
    valid_loader, 
    path_to_model_save=path2save,
    step_batch_log = 10,
    early_stopping_epoch = 5,
    logger = train_logger,
    device = DEVICE, 
    epochs = 20,
    seed = SEED
)

train: 9it [00:08,  1.12it/s]2026-04-09 01:07:38,514 - INFO - epoch 1 | batch 10 | train_loss: 0.01107577
train: 19it [00:18,  1.04it/s]2026-04-09 01:07:48,096 - INFO - epoch 1 | batch 20 | train_loss: 0.01095001
train: 29it [00:26,  1.16it/s]2026-04-09 01:07:56,840 - INFO - epoch 1 | batch 30 | train_loss: 0.01087446
train: 33it [00:30,  1.08it/s]
valid: 8it [00:08,  1.04s/it]
Computing metrics: 100%|██████████| 8/8 [00:06<00:00,  1.21it/s]
2026-04-09 01:08:14,899 - INFO - ================= epoch end =================
2026-04-09 01:08:14,899 - INFO - epoch 1 | train_loss: 0.69984826 | valid_loss: 0.68328367 | valid balanc. acc.: 0.52865607
2026-04-09 01:08:14,899 - INFO - metrics:
{'accuracy': 0.6471774193548387, 'balanced_accuracy': 0.5286560667432441, 'f1': 0.7651006711409396, 'precision': 0.7345360824742269, 'recall': 0.7983193277310925}
2026-04-09 01:08:15,018 - INFO - save checkpoint: ../models_checkpoints/resnet_18_backbone/2026-04-09_01_07_22_classifier_layer_train_v2/checkpoin

## Part 2

In [12]:
# checkpoint load 
bast_checkpoint = "../models_checkpoints/resnet_18_backbone/2026-04-09_00_18_33_classifier_layer_train/checkpoint/best_model.pth"
checkpoint = torch.load(bast_checkpoint)
model_resnet.load_state_dict(checkpoint['state_dict'])

model_resnet.to(DEVICE)

# unfreeze all parameters 
for param in model_resnet.parameters():
    param.requires_grad = True

In [13]:
NEW_LR = 1e-5

In [14]:
optimizer = optim.SGD(
    model_resnet.parameters(), 
    lr=NEW_LR,
    momentum=MOMENTUM, 
    weight_decay=LAMBDA_LR
)

In [16]:
train_logger.info(f""" 
{"=" * 50}
continue training, part 2
unfreeze all parameters
use best checkpoint model: {bast_checkpoint}

LR: {NEW_LR}
""")

2026-04-09 00:35:47,272 - INFO -  
continue training, part 2
unfreeze all parameters
use best checkpoint model: ../models_checkpoints/resnet_18_backbone/2026-04-09_00_18_33_classifier_layer_train/checkpoint/best_model.pth

LR: 1e-05



In [17]:
train_history = train_loop(
    model_resnet, 
    loss_fn, 
    optimizer, 
    train_loader, 
    valid_loader, 
    path_to_model_save=path2save,
    step_batch_log = 10,
    early_stopping_epoch = 7,
    logger = train_logger,
    device = DEVICE, 
    epochs = 20,
    seed = SEED
)

train: 9it [00:11,  1.36s/it]2026-04-09 00:36:31,343 - INFO - epoch 1 | batch 10 | train_loss: 0.00909724
train: 19it [00:29,  1.70s/it]2026-04-09 00:36:48,540 - INFO - epoch 1 | batch 20 | train_loss: 0.00902953
train: 29it [00:38,  1.03it/s]2026-04-09 00:36:57,977 - INFO - epoch 1 | batch 30 | train_loss: 0.00896304
train: 33it [00:43,  1.31s/it]
valid: 8it [00:08,  1.02s/it]
Computing metrics: 100%|██████████| 8/8 [00:06<00:00,  1.16it/s]
2026-04-09 00:37:16,611 - INFO - ================= epoch end =================
2026-04-09 00:37:16,611 - INFO - epoch 1 | train_loss: 0.56900394 | valid_loss: 0.68195020 | valid balanc. acc.: 0.61989400
2026-04-09 00:37:16,611 - INFO - metrics:
{'accuracy': 0.6330645161290323, 'balanced_accuracy': 0.619894000765774, 'f1': 0.718266253869969, 'precision': 0.8027681660899654, 'recall': 0.6498599439775911}
2026-04-09 00:37:16,738 - INFO - save checkpoint: ../models_checkpoints/resnet_18_backbone/2026-04-09_00_18_33_classifier_layer_train/checkpoint/bes